# Learn
In this guided project, we'll work with a dataset of used cars from eBay Kleinanzeigen, a classifieds section of the German eBay website.

The dataset was originally scraped and uploaded to Kaggle by user orgesleka.
The original dataset isn't available on Kaggle anymore, but you can find it here.

We've made a few modifications from the original dataset:

We sampled 50,000 data points from the full dataset, to ensure your code runs quickly in our hosted environment.
We dirtied the dataset a bit to more closely resemble what you would expect from a scraped dataset (the version uploaded to Kaggle was cleaned to be easier to work with).
Data dictionary for the dataset:

- dateCrawled - When the ad was first crawled. All field-values are taken from this date.
- name - Name of the car.
- seller - Whether the seller is private or a dealer.
- offerType - The type of listing.
- price - The listed selling price of the car.
- abtest - Whether the listing is included in an A/B test.
- vehicleType - The type of vehicle.
- yearOfRegistration - The year in which the car was first registered.
- gearbox - The type of transmission.
- powerPS - The power of the car in PS.
- model - The car model name.
- odometer - How many kilometers the car has driven.
- monthOfRegistration - The month in which the car was first registered.
- fuelType - What type of fuel the car uses.
- brand - The brand of the car.
- notRepairedDamage - If the car has a damage which is not yet repaired.
- dateCreated - The date the eBay listing was created.
- nrOfPictures - The number of pictures in the ad.
- postalCode - The postal code for the location of the vehicle.
- lastSeenOnline - When the crawler saw this ad last online.

The aim of this project is to clean the dataset and perform some initial analysis on it. While working on this project, you'll become familiar with some of the unique benefits the JupyterLab environment has to offer for pandas.

Let's start by importing the required libraries, then load the dataset using pandas.

In [ ]:
import pandas as pd
import numpy as np

In [ ]:
autos = pd.read_csv("raw_data/P3/autos.csv", encoding= "latin1")
autos.head()
autos.info()
autos.columns

# Observations
There are 20 columns where 5 of the columns have null values in them. These will be dealt with later.
Most of the data types are object type, some are int64.

There are 50,000 entries. 

In [ ]:
renames_dictionary = {
    'dateCrawled' : 'date_crawled', 
    'name' : 'name', 
    'seller' : 'seller', 
    'offerType' : 'offer_type', 
    'price' : 'price', 
    'abtest' : 'abtest',
    'vehicleType' : 'vehicle_type', 
    'yearOfRegistration' : 'registration_year', 
    'gearbox' : 'gearbox', 
    'powerPS' : 'power_ps', 
    'model' : 'model',
    'odometer' : 'odometer', 
    'monthOfRegistration' : 'registratioon_month', 
    'fuelType' : 'fuel_type', 
    'brand' : 'brand',
    'notRepairedDamage' : 'unrepaired_damange', 
    'dateCreated' : 'ad_created', 
    'nrOfPictures' : 'nr_of_pictures', 
    'postalCode' : 'postal_code',
    'lastSeen' : 'last_seen'
}

autos = autos.rename(columns = renames_dictionary)
autos.info()
autos.head()

# Changes made
We made a few changes to the column names, I believe the best way to name the columns is to use underscores instead of spaces, and snake case. Since making it all lower case would convert from the camel case to the sname case but readibility would suffer due to no spaces between the words. 

In [ ]:
autos.describe(include='all')

In [ ]:
autos['offer_type'].value_counts()

# changes to be made
The following list tabulates what should be modified in cleaning of the data. 

- oodmeter has a Km written in the value and the rpice column has a $ sign (that will be transfered to column name and removed from the individual value in the column)
- 
- seller has no information contained as all but one value are listed privet. Either the information is irrelevant or not granualar enough to differentiate between the privet values.
-
- offer_type columns as well contains no real information all but one value is differnet.
- The rpice column has a weird value that there are 1421 values with $0 price. This might not be an anomaly as some cars might be acutally just donated and up for grabs by anyone free of cost.
- There seems to be a abtest column with an equal separation in a and b, I am unaware what this abtesting was (will read the documentation)
- Anomaly in the year of registration, one value is mislabled for sure as the vehicle registration was made in 1001, either it was 2001 or something else. but we will leave it as it is, but when we perform analysis on this column, we will remove that entry.
- there are about 20% values that do not have a registration month associated with them (value is set to 0)
- Unrepaired damage is a binary in german, we can convert the vlues to yes and no for ja and nein, if we want, but it is understandable for me.
- nr_of_pictures column is all 0 so useless, toss it out.

Every other column seems to be good. 

In [ ]:
# remove non numeric values in columns of price and odometer

autos['price'] = autos['price'].str.replace('$', '').str.replace(',', '').astype('int64')

autos['odometer'] = autos['odometer'].str.replace('km', '').str.replace(',', '').astype('int64')


new_col_dict = {
    'price' : 'price_in_dollar', 
    'odometer' : 'odometer_km'
}
autos = autos.rename(columns = new_col_dict)


In [ ]:
# remove the colums that contain invariant information

autos.drop(['seller', 'offer_type', 'nr_of_pictures'], axis =1).info()

In [ ]:
print(
    autos['price_in_dollar'].unique().shape,
    autos['price_in_dollar'].describe(),
    autos['price_in_dollar'].value_counts().sort_index().head(20)
)

In [ ]:
print(
    autos['odometer_km'].unique().shape,
    autos['odometer_km'].describe(),
    autos['odometer_km'].value_counts().sort_index()
)

# Issues with the price column, outliers
There are certain values that are too many. I decided to remove all the values that are greater than 900,000 as those are more than an order of magnitude higher than the previous highest value. 

There remain alot of values that are 0 for the price, I believe that is possible if the car is truely worthless. So I let that be in the dataset. 


In [ ]:
autos = autos.loc[autos['price_in_dollar'] < 900000]

In [ ]:
autos['date_crawled'].describe()

In [ ]:
print(
    autos['date_crawled'].str[:10].value_counts(normalize=True, dropna=False).sort_index(),
    autos['ad_created'].str[:10].value_counts(normalize=True, dropna=False).sort_index(),
    autos['last_seen'].str[:10].value_counts(normalize=True, dropna=False).sort_index(),
    autos['registration_year'].describe(include='all'),
    autos['registration_year'].value_counts().sort_index()
)

In [ ]:
autos = autos[autos['registration_year'].between(1900, 2016)]

# remove outliers from the registration data
Remove vehicles that are registed after they were listed on the ebay for sale, and remove ancient values, before cars were athing i.e. before 20th century
We removed a lot of values by this filtering, final entries reomved now stand at about 2000. 

In [ ]:
print(
    autos['brand'].unique().shape,
    autos['brand'].describe(),
    autos['brand'].value_counts().sort_values()
)

In [ ]:
brand_index = autos['brand'].value_counts().index
print(brand_index.shape)

In [ ]:
mean_price_brand = {}

for value in brand_index[0:8]:
    subset_data = autos.loc[autos['brand'] == value] 
    mean_price_brand[value] = subset_data['price_in_dollar'].mean()

print(mean_price_brand)


In [ ]:
mean_price = {}
mean_milage = {}
mean_registration = {}

for value in brand_index[0:8]:
    subset_data = autos.loc[autos['brand'] == value] 
    mean_price[value] = subset_data['price_in_dollar'].mean()
    mean_milage[value] = subset_data['odometer_km'].mean()
    mean_registration[value] = subset_data['registration_year'].mean()

print(mean_price, mean_milage, mean_registration)

In [ ]:
mean_price_df = pd.Series(mean_price).sort_values(ascending= False)
mean_milage_df = pd.Series(mean_milage)
mean_registration_df = pd.Series(mean_registration)
df = pd.DataFrame(mean_price_df, columns = ['mean_price'])
df['mean_milage'] = mean_milage_df
df['mean_registration'] = mean_registration_df
print(df)

# anaysis

While the comapnies audi, mercedes, and bmw has the highest resale value, we arent sure what the buying price of these was, if we knew that, we could have calculated the % deprecation.

What is interesting is that the almost all the cars milage is pretty compareable.

Hoever it seems like the audi cars were on the market the fastest, while the mercedes were retained by the owners for the longest time. That does not however correlate with the milage, which is pretty comparable for all the car brands. 


# Next steps

Data cleaning next steps:
- Identify categorical data that uses german words, translate them and map the values to their english counterparts
- Convert the dates to be uniform numeric data, so "2016-03-21" becomes the integer 20160321.
- See if there are particular keywords in the name column that you can extract as new columns

Analysis next steps:
- Find the most common brand/model combinations
- Split the odometer_km into groups, and use aggregation to see if average prices follows any patterns based on the mileage.
- How much cheaper are cars with damage than their non-damaged counterparts?